# WSI Stiching


In [1]:
import os
import gc
from pathlib import Path

import cv2
import numpy as np
import scipy.ndimage
import tifffile as tiff
import torch
import torch.nn.functional as F
from ultralytics import YOLO
from segment_anything import sam_model_registry

CHECKPOINT_DIR = Path(os.environ.get('HUBMAP_CHECKPOINT_DIR', './checkpoints'))
YOLO_WEIGHTS = CHECKPOINT_DIR / 'yolov11s_detector_best.pt'
MEDSAM_WEIGHTS = CHECKPOINT_DIR / 'medsam_finetuned_best.pth'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

PATCH_SIZE = 1024
STRIDE_PIXELS = 307
YOLO_CONF_THRESHOLD = 0.40
MEDSAM_PROB_THRESHOLD = 0.70
MAJORITY_VOTE_RATIO = 0.50
MIN_GLOMERULUS_AREA_PX = 15000

In [2]:
def postprocess_wsi_binary_mask(raw_binary: np.ndarray, min_area: int = MIN_GLOMERULUS_AREA_PX) -> np.ndarray:
    # Fill holes, filter noise
    labeled, _ = scipy.ndimage.label(raw_binary)
    slices = scipy.ndimage.find_objects(labeled)
    clean_mask = np.zeros_like(raw_binary, dtype=np.uint8)
    for idx, sl in enumerate(slices, 1): # for each component
        if sl is None:
            continue
        component = (labeled[sl] == idx)
        if int(component.sum()) < min_area: # Discard if component is too small
            continue
        filled = scipy.ndimage.binary_fill_holes(component)
        clean_mask[sl][filled] = 255
    return clean_mask


def run_baseline_wsi_inference(
    image_rgb: np.ndarray,
    yolo_model: YOLO,
    sam_model: torch.nn.Module,
    patch_size: int = PATCH_SIZE,
    stride: int = STRIDE_PIXELS,
    conf_thresh: float = YOLO_CONF_THRESHOLD,
    medsam_thresh: float = MEDSAM_PROB_THRESHOLD,
    vote_ratio: float = MAJORITY_VOTE_RATIO,
    min_area: int = MIN_GLOMERULUS_AREA_PX,
) -> np.ndarray:

    h, w = image_rgb.shape[:2]
    vote_accumulator = np.zeros((h, w), dtype=np.uint8)
    coverage_counter = np.zeros((h, w), dtype=np.uint8)

    for y in range(0, h, stride):
        for x in range(0, w, stride):
            y_end, x_end = min(y + patch_size, h), min(x + patch_size, w)
            ph, pw = y_end - y, x_end - x
            coverage_counter[y:y_end, x:x_end] += 1
            if ph < 32 or pw < 32:
                continue

            patch_rgb = image_rgb[y:y_end, x:x_end]
            gray = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2GRAY)
            if (np.sum(gray >= 230) / (ph * pw)) > 0.95 or (np.sum(gray <= 15) / (ph * pw)) > 0.95: # skip if mostly empty background
                continue

            patch_bgr = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2BGR)
            detections = yolo_model(patch_bgr, verbose=False, conf=conf_thresh, device=DEVICE)
            if len(detections) == 0 or len(detections[0].boxes) == 0: # skip patch if yolo didn't detect a box
                continue

            boxes = detections[0].boxes.xyxy.cpu().numpy()
            padded_rgb = np.zeros((patch_size, patch_size, 3), dtype=np.uint8)
            padded_rgb[:ph, :pw] = patch_rgb

            with torch.no_grad():
                img_tensor = torch.from_numpy(padded_rgb).to(DEVICE).permute(2, 0, 1).float().unsqueeze(0)
                with torch.amp.autocast('cuda', enabled=(DEVICE != 'cpu')):
                    embedding = sam_model.image_encoder(img_tensor)
                embedding = embedding.float()

                patch_binary = np.zeros((ph, pw), dtype=np.uint8)
                for box in boxes:
                    x1, y1 = max(0.0, float(box[0])), max(0.0, float(box[1]))
                    x2, y2 = min(float(pw), float(box[2])), min(float(ph), float(box[3]))
                    ix1, iy1, ix2, iy2 = int(x1), int(y1), int(x2), int(y2)
                    if ix2 <= ix1 or iy2 <= iy1:
                        continue
                    box_tensor = torch.tensor([[x1, y1, x2, y2]], device=DEVICE, dtype=torch.float32)
                    sparse_emb, dense_emb = sam_model.prompt_encoder(points=None, boxes=box_tensor, masks=None)
                    low_res_masks, _ = sam_model.mask_decoder(
                        image_embeddings=embedding,
                        image_pe=sam_model.prompt_encoder.get_dense_pe(),
                        sparse_prompt_embeddings=sparse_emb,
                        dense_prompt_embeddings=dense_emb,
                        multimask_output=False,
                    )
                    upsampled = F.interpolate(
                        low_res_masks, size=(patch_size, patch_size), mode='bilinear', align_corners=False
                    )
                    prob_crop = torch.sigmoid(upsampled[0, 0, iy1:iy2, ix1:ix2]).cpu().numpy()
                    patch_binary[iy1:iy2, ix1:ix2] = np.maximum(
                        patch_binary[iy1:iy2, ix1:ix2], (prob_crop > medsam_thresh).astype(np.uint8)
                    )

                vote_accumulator[y:y_end, x:x_end] += patch_binary

    raw_majority = (
        (vote_accumulator.astype(np.uint16) * int(round(1.0 / vote_ratio)))
        >= coverage_counter.astype(np.uint16)
    ) & (coverage_counter > 0)
    return postprocess_wsi_binary_mask(raw_majority, min_area=min_area)

In [3]:
import json
import time

DATA_ROOT = Path(os.environ.get('HUBMAP_DATA_ROOT', './data/hubmap_pas_wsis'))
SPLIT_CONFIG_PATH = Path(os.environ.get('HUBMAP_SPLIT_CONFIG', './data/dataset_split.json'))

with open(SPLIT_CONFIG_PATH, 'r', encoding='utf-8') as f:
    split_cfg = json.load(f)

ome_registry = {}
for d in sorted(DATA_ROOT.iterdir()):
    if d.is_dir() and d.name.startswith('HBM'):
        for p in d.rglob('*.ome_mask.json'):
            van_prefix = p.name.split('_PAS')[0].split('-PAS')[0]
            ome_registry[van_prefix] = p

yolo_model = YOLO(str(YOLO_WEIGHTS)).to(DEVICE)
sam_model = sam_model_registry['vit_b'](checkpoint=str(MEDSAM_WEIGHTS)).to(DEVICE).eval()

print('Running Full-WSI Sliding-Window Inference (patch_size=1024, stride=307, 70% overlap, Standard Majority Voting >= 50%)...')
wsi_dices, wsi_ious, wsi_precs, wsi_recs = [], [], [], []

for idx, slide_folder in enumerate(split_cfg['test_ids'], 1):
    slide_dir = DATA_ROOT / slide_folder
    short_id = slide_folder.split('-')[0]
    tiff_path = [p for p in slide_dir.rglob('*.tif*') if 'mask' not in p.name.lower()][0]
    local_ome = list(slide_dir.rglob('*.ome_mask.json'))
    van_prefix = tiff_path.name.split('_PAS')[0].split('-PAS')[0]
    ome_paths = [local_ome[0] if local_ome else ome_registry[van_prefix]]
    extra_jsons = [p for p in slide_dir.rglob('*glomerulus-annotations.json')]
    if short_id in ('HBM662.PMPZ.644', 'HBM344.LLLV.539') and extra_jsons:
        ome_paths.extend(extra_jsons)

    try:
        image_rgb = np.squeeze(tiff.memmap(str(tiff_path)))
    except Exception:
        image_rgb = np.squeeze(tiff.imread(str(tiff_path)))
    while image_rgb.ndim > 3:
        image_rgb = np.squeeze(image_rgb[0])
    if image_rgb.ndim == 3 and image_rgb.shape[0] in (1, 3):
        image_rgb = np.transpose(image_rgb, (1, 2, 0))
    if image_rgb.ndim == 3 and image_rgb.shape[-1] > 3:
        image_rgb = image_rgb[:, :, :3]
    h, w = image_rgb.shape[:2]

    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for jp in ome_paths:
        with open(jp, 'r', encoding='utf-8') as jf:
            gj = json.load(jf)
        feats = gj if isinstance(gj, list) else gj.get('features', [])
        for feat in feats:
            coords = feat.get('geometry', {}).get('coordinates', [])
            if coords:
                cv2.fillPoly(gt_mask, [np.array(coords[0], dtype=np.int32)], 255)

    pred_mask = run_baseline_wsi_inference(image_rgb, yolo_model, sam_model)

    pred_bin = pred_mask > 127
    gt_bin = gt_mask > 127
    tp = int(np.logical_and(pred_bin, gt_bin).sum())
    fp = int(pred_bin.sum()) - tp
    fn = int(gt_bin.sum()) - tp
    dice = (2.0 * tp) / (2.0 * tp + fp + fn + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)
    prec = tp / (tp + fp + 1e-6)
    rec = tp / (tp + fn + 1e-6)
    wsi_dices.append(dice)
    wsi_ious.append(iou)
    wsi_precs.append(prec)
    wsi_recs.append(rec)
    print(f'  [{idx}/7] {short_id} ({h}x{w}) -> Foreground Dice = {dice:.4f} (IoU = {iou:.4f}, Precision = {prec:.4f}, Sensitivity = {rec:.4f})', flush=True)
    del image_rgb, gt_mask, pred_mask, pred_bin, gt_bin
    gc.collect()

print(f'Completed 7/7 Test WSIs | Mean WSI Foreground Dice = {np.mean(wsi_dices):.4f} +/- {np.std(wsi_dices):.4f} (IoU = {np.mean(wsi_ious):.4f} +/- {np.std(wsi_ious):.4f}, Precision = {np.mean(wsi_precs):.4f}, Sensitivity = {np.mean(wsi_recs):.4f})')

Running Full-WSI Sliding-Window Inference (patch_size=1024, stride=307, 70% overlap, Standard Majority Voting >= 50%)...
  [1/7] HBM649.DLZF.463 (20329x33103) -> Foreground Dice = 0.9670 (IoU = 0.9361, Precision = 0.9506, Sensitivity = 0.9840)
  [2/7] HBM662.PMPZ.644 (26840x49780) -> Foreground Dice = 0.9046 (IoU = 0.8258, Precision = 0.9396, Sensitivity = 0.8721)
  [3/7] HBM833.DBGG.252 (23990x47723) -> Foreground Dice = 0.9744 (IoU = 0.9500, Precision = 0.9602, Sensitivity = 0.9890)
  [4/7] HBM344.LLLV.539 (22153x36732) -> Foreground Dice = 0.9511 (IoU = 0.9068, Precision = 0.9260, Sensitivity = 0.9776)
  [5/7] HBM673.JJRZ.435 (22134x18491) -> Foreground Dice = 0.9362 (IoU = 0.8801, Precision = 0.8909, Sensitivity = 0.9864)
  [6/7] HBM227.THVC.544 (30368x26624) -> Foreground Dice = 0.9709 (IoU = 0.9435, Precision = 0.9557, Sensitivity = 0.9866)
  [7/7] HBM627.RSGW.898 (30400x28672) -> Foreground Dice = 0.9669 (IoU = 0.9360, Precision = 0.9505, Sensitivity = 0.9839)
Completed 7/7 Test